In [1]:
#%pip install transformers
#%pip install "transformers[torch]"

In [1]:
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments
)

In [3]:
import pandas as pd

In [8]:
train_data = pd.read_csv("./Dataset/samsum_train.csv")
val_data = pd.read_csv("./Dataset/samsum_validation.csv")

In [ ]:
#train_data.head()

In [10]:
# Random sampling 
train_data = train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data = train_data.sample(n=1000,random_state=42).reset_index(drop=True)

In [11]:
train_data.shape

(4000, 3)

In [12]:
val_data.shape

(1000, 3)

# Data Pre-processing

In [13]:
import re

def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags <p> <h1>
    text = text.strip().lower()
    return text

In [14]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [ ]:
# train_data["dialogue"][0]

## Tokenize

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [19]:
# raw data => tokenized inputs  for fine-tuning

def tokenize(data):
    inputs  = tokenizer(data["dialogue"], padding="max_length", max_length=512 , truncation=True)
    targets = tokenizer(data["summary"],  padding="max_length", max_length=150 , truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids = add to input as labels 
    return inputs 

In [21]:
train_dataset = train_data.apply(tokenize,axis=1).tolist()
val_dataset = val_data.apply(tokenize,axis=1).tolist()

In [ ]:
# train_dataset[0]

In [23]:
# input ids - dialogue

# 1 -> EOS  , 0 -> padding

# attention mask
# labels - target => summary token

In [27]:
print(len(train_dataset[0]["input_ids"]))
print(type(train_dataset))
print(type(val_dataset))

512
<class 'list'>
<class 'list'>


# Working with Our Model

In [31]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [32]:
# NLP => generation task

model = T5ForConditionalGeneration.from_pretrained("t5-small")
model = model.to(device)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [35]:
# Training Arguments 

training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs=6,
    weight_decay=0.1,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy = "epoch",
    save_strategy = "epoch" ,
    warmup_steps = 500
    # 0 => lr default
)

In [37]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset
)

In [39]:
# train the model
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 